# 🚀 Fine-Tuning Qwen2.5-0.5B on Full 126,617 Multilingual RAG Dataset (Hindi, Tamil, English)

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** using QLoRA on **100% of all passages** in the Hindi, Tamil, and English corpus (~126,617 examples).

### Estimated Time on Free Colab T4 GPU: **~20 to 30 minutes**

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes

In [ ]:
# Step 2: Check Uploaded Dataset (rag_sft_dataset.jsonl ~160 MB)
import os
if not os.path.exists('rag_sft_dataset.jsonl'):
    print('Please upload rag_sft_dataset.jsonl in the Colab Files panel on the left.')
else:
    size_mb = os.path.getsize('rag_sft_dataset.jsonl') / (1024 * 1024)
    print(f'✅ Dataset found! Size: {size_mb:.2f} MB')

In [ ]:
# Step 3: Run Full Multilingual Fine-Tuning
import json, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DATASET_PATH = 'rag_sft_dataset.jsonl'
OUTPUT_DIR = 'qwen2.5_0.5b_indic_rag_lora'
MERGED_DIR = 'qwen2.5_0.5b_indic_rag_merged'

# Load dataset
print('Loading 126,617 examples...')
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
dataset = dataset.shuffle(seed=42).train_test_split(test_size=0.03)
print(f'Train split: {len(dataset["train"])}, Eval split: {len(dataset["test"])}')

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}

formatted_train = dataset['train'].map(format_chat_template)
formatted_eval = dataset['test'].map(format_chat_template)

# Load 4-bit quantized base model
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

# LoRA Configuration
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Optimized Training Arguments for 126k dataset on T4 GPU
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    num_train_epochs=1,
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=1000,
    save_strategy='steps',
    save_steps=1000,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    save_total_limit=1
)

trainer = SFTTrainer(model=model, train_dataset=formatted_train, eval_dataset=formatted_eval, dataset_text_field='text', max_seq_length=384, tokenizer=tokenizer, args=training_args)
trainer.train()

In [ ]:
# Step 4: Merge LoRA Weights and Export Standalone Model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('Merging LoRA weights with base model...')
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
from peft import PeftModel
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR).merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Zip the merged model for 1-click download
!zip -r qwen2.5_0.5b_indic_rag_merged.zip qwen2.5_0.5b_indic_rag_merged
print('✅ Merged standalone model zipped as: qwen2.5_0.5b_indic_rag_merged.zip!')

In [ ]:
# Step 5 (Optional): Push directly to your Hugging Face Account
# from huggingface_hub import login
# login() # Paste your HF token
# merged_model.push_to_hub('ansh123456789/qwen2.5-0.5b-indic-rag')
# tokenizer.push_to_hub('ansh123456789/qwen2.5-0.5b-indic-rag')